# Final SELF-DISCOVER v6c compact-plan runner

This is the release runner for the final SELF-DISCOVER condition only. Set `DATASET` to `reddit` or `mixed_emotion` and `MODEL` to `llama2` or `llama3`. The model-specific cached plan is reused across both datasets. `RUN_VARIANTS` is fixed to `v6c_compact`; exploratory variants are excluded.


In [ ]:
# Run once in a fresh Colab GPU runtime. Restart the runtime after installation.
%pip install -q pandas==2.2.3 numpy==2.1.3 scipy==1.16.3 scikit-learn==1.6.1 accelerate==1.14.0 transformers==5.17.0 bitsandbytes==0.50.2 sentencepiece protobuf tqdm

import importlib.metadata as importlib_metadata
print("bitsandbytes:", importlib_metadata.version("bitsandbytes"))
print("transformers:", importlib_metadata.version("transformers"))
print("accelerate:", importlib_metadata.version("accelerate"))
print("\nRestart the runtime once, then run from the imports cell.")

In [ ]:
MIXED_EMOTION_PAYLOAD_B64 = (
    "..."
)

In [ ]:
import gc
import hashlib
import json
import os
import re
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from IPython.display import display
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

LABELS = ["Depression", "Neutral", "Happy"]
LABEL_SET = set(LABELS)

## Configuration and final Phase 1 input contract

Choose one dataset and one model. Hash and row-count guards stop incompatible inputs before inference.


In [ ]:
# Which dataset to run. "reddit" is the 12,000-row held-out Reddit test set; "mixed_emotion" is
# the 300-example synthetic stress test. Everything downstream keys off this.
DATASET = "reddit"

# Per-dataset contract. The hashes are SHA-256 over the sorted routed example_id values, and they
# identify the exact routed sets used by the final comparison. A mismatch fails the run.
DATASET_CONFIG = {
    "reddit": {
        "input_source": "drive",
        "expected_total_rows": 12_000,
        "expected_routed_rows": 218,
        "routed_id_hash": "e852273108039cebe31ef965ad0cea484d9a61835e1b608de6fe1f1a6becaa88",
        "output_subdir": "phase2",
    },
    "mixed_emotion": {
        "input_source": "embedded",
        "expected_total_rows": 300,
        "expected_routed_rows": 86,
        "routed_id_hash": "a4c06972fac45e51a989f378d7daad9d64ab1354113b834a770bfd2c33df18f6",
        "output_subdir": "mixed_emotion/phase2",
    },
}
if DATASET not in DATASET_CONFIG:
    raise ValueError(f"DATASET must be one of {sorted(DATASET_CONFIG)}")
ACTIVE_DATASET = DATASET_CONFIG[DATASET]

# Select the Llama version. Each model uses its own released compact plan.
MODEL = "llama3"

MODEL_CONFIG = {
    "llama3": {
        "name": "NousResearch/Meta-Llama-3-8B-Instruct",
        "revision": "53346005fb0ef11d3b6a83b12c895cca40156b6c",
        "context_tokens": 8192,
        "max_new_tokens": 1024,        # per stage, for the multi-turn variants
        "v6_max_new_tokens": 1536,     # v6 carries all its reasoning in one pass
        "folder": "llama3_self_discover_advanced",
    },
    "llama2": {
        # 4096-token context, so both the input budget and the generation cap are tighter.
        "name": "NousResearch/Llama-2-7b-chat-hf",
        "revision": "351844e75ed0bcbbe3f10671b3c808d2b83894ee",
        "context_tokens": 4096,
        "max_new_tokens": 512,         # matches 05_1's recorded Llama 2 CoT condition
        "v6_max_new_tokens": 1024,     # Llama 3 v6 averaged 416 generated tokens, so this is ample
        "folder": "llama2_self_discover_advanced",
    },
}
if MODEL not in MODEL_CONFIG:
    raise ValueError(f"MODEL must be one of {sorted(MODEL_CONFIG)}")
ACTIVE_MODEL = MODEL_CONFIG[MODEL]

# Each running model uses its own released compact plan.
STRUCTURE_SOURCE_MODEL = MODEL
PHASE1_INPUT_SOURCE = ACTIVE_DATASET["input_source"]
EXPECTED_TOTAL_ROWS = ACTIVE_DATASET["expected_total_rows"]
EXPECTED_ROUTED_ROWS = ACTIVE_DATASET["expected_routed_rows"]
EXPECTED_ROUTED_ID_HASH = ACTIVE_DATASET["routed_id_hash"]

# Reddit Phase 2 input contains original post text and is intentionally not published.
# The default reads the authorized Drive export. An authorized URL can be supplied explicitly
# through CGSLR_PHASE1_INPUT_URL and selected by setting input_source to "url" above.
# Mixed Emotion remains embedded in this output-free notebook for exact public reproduction.
PHASE1_INPUT_URL = os.environ.get("CGSLR_PHASE1_INPUT_URL")
# Used when PHASE1_INPUT_SOURCE == "drive"; identical to the path 05_2 consumed.
PHASE1_INPUT_PATH = Path(
    "/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/"
    "unified_final/phase1/final_phase1_reasoning_input.csv"
)
LOCAL_INPUT_CACHE = Path(f"/content/phase1_reasoning_input_{DATASET}.csv")

# Drive holds the row-level checkpoints that make a multi-hour run resumable. Losing it means
# losing every completed row on a disconnect, so a failed mount is an error by default.
# Set this to False only when running somewhere with durable local storage.
REQUIRE_DRIVE = True
LOCAL_OUTPUT_ROOT = Path("outputs_final/unified_final/phase2")
UNIFIED_FINAL_ROOT = Path(
    "/content/drive/MyDrive/confidence_guided_llm_reasoning/outputs_final/unified_final"
)
DRIVE_OUTPUT_ROOT = UNIFIED_FINAL_ROOT / ACTIVE_DATASET["output_subdir"]

# Frozen final Phase 1 contract. Any accidental input change fails before inference.


MAX_ROWS = None  # None for the final run; use a small integer (e.g. 8) for a smoke test.
MAX_REASONING_CHARACTERS = 6000
REASONING_HEAD_CHARACTERS = 3500
REASONING_TAIL_CHARACTERS = 2500
LOAD_IN_4BIT = True
RESUME_FROM_EXISTING = True
BASE_SEED = 42

RUN_VARIANTS = ["v6c_compact"]



# Canonical required columns in PHASE1_INPUT_PATH:
# example_id, target_label, phase1_label, phase1_confidence,
# phase1_routed, phase2_original_text
# title + selftext may replace phase2_original_text; aliases below are normalized.

In [ ]:
MOUNT_GUIDANCE = {
    "not_colab": (
        "  1. Confirm this is a Colab runtime. Local Jupyter, VS Code and Kaggle have no\n"
        "     google.colab module, which is exactly what the reason above reports.\n"
        "  2. If you meant to run outside Colab, set REQUIRE_DRIVE = False (see below)."
    ),
    "credentials": (
        "  This is a browser-side authorisation failure, not a problem with this notebook.\n"
        "  Colab could not carry your Google credentials back into the runtime.\n"
        "  In rough order of how often each one fixes it:\n"
        "  1. Allow third-party cookies for accounts.google.com and googleusercontent.com.\n"
        "     Chrome: Settings > Privacy and security > Third-party cookies > Allow, or add\n"
        "     those two sites as exceptions. Blocked third-party cookies are the usual cause.\n"
        "  2. Turn off ad blockers and privacy extensions for colab.research.google.com, and\n"
        "     use a normal window rather than Incognito or a strict-privacy browser.\n"
        "  3. Sign out of every Google account except the one that owns the Drive folder, then\n"
        "     hard-reload the page. Multiple signed-in accounts often break the handshake.\n"
        "  4. Runtime > Disconnect and delete runtime, reload the page, run this cell again,\n"
        "     and approve the prompt promptly. It expires if left unattended.\n"
        "  5. Mount from the UI instead: Files panel (folder icon on the left) > Mount Drive.\n"
        "     Once it succeeds there, re-run this cell.\n"
        "  6. Try a different browser."
    ),
    "stale_mount": (
        "  1. Runtime > Disconnect and delete runtime, then reconnect. This clears a stale\n"
        "     /content/drive mountpoint, which a force_remount could not recover here.\n"
        "  2. Then run this cell again and approve the Google account prompt."
    ),
    "unknown": (
        "  1. Run this cell again and approve the Google account prompt. It expires if a cell\n"
        "     is left unattended.\n"
        "  2. Runtime > Disconnect and delete runtime, then reconnect and retry.\n"
        "  3. Mount from the Files panel (folder icon on the left) > Mount Drive."
    ),
}


def mount_drive():
    '''Mount Drive. Returns (None, None) on success, else (kind, reason).'''
    try:
        from google.colab import drive
    except Exception as exc:
        return "not_colab", f"this is not a Google Colab runtime ({type(exc).__name__}: {exc})"
    try:
        drive.mount("/content/drive")
        return None, None
    except Exception as first:
        reason = f"{type(first).__name__}: {first}"
        text = f"{type(first).__name__} {first}".lower()
        if "must not already contain files" in text or "already mounted" in text:
            # A stale mountpoint is recoverable; a forced remount usually clears it.
            try:
                drive.mount("/content/drive", force_remount=True)
                print("Drive re-mounted after a stale mountpoint.")
                return None, None
            except Exception as second:
                return "stale_mount", f"{reason} (force_remount also failed: {second})"
        if "credential" in text or "messageerror" in text or "access_denied" in text:
            # Retrying only triggers a second prompt that fails the same way.
            return "credentials", reason
        return "unknown", reason


drive_failure_kind, drive_failure = mount_drive()
if drive_failure is None:
    print(f"Drive mounted. Outputs go to {DRIVE_OUTPUT_ROOT}")
elif REQUIRE_DRIVE:
    raise RuntimeError(
        "Could not mount Google Drive, so the run would not survive a disconnect.\n"
        f"Reason: {drive_failure}\n\n"
        f"{MOUNT_GUIDANCE[drive_failure_kind]}\n\n"
        "  Last resort: set REQUIRE_DRIVE = False in the config cell. Results then go to "
        f"{LOCAL_OUTPUT_ROOT}\n"
        "  and are LOST when the runtime ends. For a multi-hour run that means losing "
        "everything\n  generated so far, so download the CSVs before disconnecting."
    )
else:
    DRIVE_OUTPUT_ROOT = LOCAL_OUTPUT_ROOT
    print(f"WARNING: no Drive ({drive_failure}).")
    print(f"WARNING: writing to {DRIVE_OUTPUT_ROOT}. These files are lost when the runtime ends.")
    print("WARNING: copy them out before disconnecting, or the run cannot be resumed.")

try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
except Exception:
    hf_token = os.environ.get("HF_TOKEN")
    if not hf_token:
        print("HF_TOKEN is not set; continuing with public Hugging Face access.")
if hf_token:
    os.environ["HF_TOKEN"] = hf_token
    os.environ["HUGGINGFACE_HUB_TOKEN"] = hf_token

DRIVE_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


def resolve_phase1_input():
    '''Return the local path of the Phase 2 input export, downloading it when needed.'''
    if PHASE1_INPUT_SOURCE == "drive":
        if not PHASE1_INPUT_PATH.exists():
            raise FileNotFoundError(f"Final Phase 1 export not found: {PHASE1_INPUT_PATH}")
        print(f"Phase 1 input (drive): {PHASE1_INPUT_PATH}")
        return PHASE1_INPUT_PATH
    if PHASE1_INPUT_SOURCE == "embedded":
        import base64
        import zlib
        decoded = zlib.decompress(base64.b64decode(MIXED_EMOTION_PAYLOAD_B64))
        LOCAL_INPUT_CACHE.parent.mkdir(parents=True, exist_ok=True)
        frame = pd.DataFrame(json.loads(decoded.decode("utf-8")))
        frame.to_csv(LOCAL_INPUT_CACHE, index=False)
        print(f"Phase 1 input (embedded): {LOCAL_INPUT_CACHE}")
        print(f"Payload SHA-256: {hashlib.sha256(decoded).hexdigest()}")
        return LOCAL_INPUT_CACHE
    if PHASE1_INPUT_SOURCE != "url":
        raise ValueError("PHASE1_INPUT_SOURCE must be 'url', 'drive' or 'embedded'.")
    if not PHASE1_INPUT_URL:
        raise ValueError(
            "PHASE1_INPUT_SOURCE='url' requires the CGSLR_PHASE1_INPUT_URL environment variable."
        )

    import urllib.request

    if not LOCAL_INPUT_CACHE.exists() or LOCAL_INPUT_CACHE.stat().st_size == 0:
        LOCAL_INPUT_CACHE.parent.mkdir(parents=True, exist_ok=True)
        print(f"Downloading {PHASE1_INPUT_URL}")
        urllib.request.urlretrieve(PHASE1_INPUT_URL, LOCAL_INPUT_CACHE)
    digest = hashlib.sha256(LOCAL_INPUT_CACHE.read_bytes()).hexdigest()
    print(f"Phase 1 input (configured URL): {LOCAL_INPUT_CACHE}")
    print(f"File SHA-256: {digest}")
    return LOCAL_INPUT_CACHE


RESOLVED_PHASE1_INPUT = resolve_phase1_input()


def normalize_bool(value):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {"true", "1", "yes", "y"}:
        return True
    if text in {"false", "0", "no", "n"}:
        return False
    raise ValueError(f"Unrecognized Boolean value: {value!r}")


def normalize_label(value):
    text = str(value).strip().lower()
    for label in LABELS:
        if text == label.lower():
            return label
    return np.nan


def minimally_sanitize_original_text(title, selftext):
    title = "" if pd.isna(title) else str(title).strip()
    body = "" if pd.isna(selftext) else str(selftext).strip()
    if body.lower() in {"[deleted]", "[removed]", "nan", "none"}:
        body = ""
    combined = title if not body else f"{title}\n\n{body}"
    combined = re.sub(r"https?://\S+|www\.\S+", " [URL] ", combined, flags=re.I)
    combined = re.sub(r"(?<!\w)(?:/u/|u/)[A-Za-z0-9_-]+", "[USER]", combined)
    combined = combined.replace("\r\n", "\n").replace("\r", "\n")
    combined = re.sub(r"[ \t]+", " ", combined)
    return re.sub(r"\n{3,}", "\n\n", combined).strip()


def limit_reasoning_text(value):
    value = "" if pd.isna(value) else str(value)
    if len(value) <= MAX_REASONING_CHARACTERS:
        return value, False
    marker = "\n\n[Middle omitted to fit model context; beginning and ending preserved.]\n\n"
    available = MAX_REASONING_CHARACTERS - len(marker)
    head_characters = min(REASONING_HEAD_CHARACTERS, available)
    tail_characters = min(REASONING_TAIL_CHARACTERS, available - head_characters)
    return value[:head_characters] + marker + value[-tail_characters:], True


def normalize_phase1_export(path):
    frame = pd.read_csv(path)
    aliases = {
        "target_label": ["target_label", "label_str", "true_label", "reference_label"],
        "phase1_label": ["phase1_label", "prediction", "predicted_label"],
        "phase1_confidence": ["phase1_confidence", "calibrated_confidence", "confidence"],
        "phase1_routed": ["phase1_routed", "routed", "route_indicator"],
        "phase2_original_text": ["phase2_original_text", "original_text", "reasoning_text"],
    }
    for canonical, candidates in aliases.items():
        if canonical in frame.columns:
            continue
        found = next((candidate for candidate in candidates if candidate in frame.columns), None)
        if found:
            frame = frame.rename(columns={found: canonical})

    if "phase2_original_text" not in frame.columns:
        if {"title", "selftext"}.issubset(frame.columns):
            frame["phase2_original_text"] = [
                minimally_sanitize_original_text(title, body)
                for title, body in zip(frame["title"], frame["selftext"])
            ]
        else:
            raise ValueError(
                "Phase 1 export needs phase2_original_text, or both title and selftext."
            )

    required = {
        "example_id", "target_label", "phase1_label", "phase1_confidence",
        "phase1_routed", "phase2_original_text",
    }
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f"Final Phase 1 export is missing columns: {sorted(missing)}")

    frame["example_id"] = frame["example_id"].astype(str)
    if frame["example_id"].duplicated().any():
        duplicates = frame.loc[frame["example_id"].duplicated(), "example_id"].head().tolist()
        raise ValueError(f"Duplicate example_id values found: {duplicates}")

    frame["target_label"] = frame["target_label"].map(normalize_label)
    frame["phase1_label"] = frame["phase1_label"].map(normalize_label)
    frame["phase1_routed"] = frame["phase1_routed"].map(normalize_bool)
    frame["phase1_confidence"] = pd.to_numeric(frame["phase1_confidence"], errors="raise")
    if frame[["target_label", "phase1_label"]].isna().any().any():
        raise ValueError("target_label and phase1_label must use Depression, Neutral, or Happy.")
    if not frame["phase1_confidence"].between(0, 1).all():
        raise ValueError("phase1_confidence must be between 0 and 1.")

    if "phase2_input_was_truncated" in frame.columns:
        previously_truncated = (
            frame["phase2_input_was_truncated"].fillna(False).map(normalize_bool)
        )
    else:
        previously_truncated = pd.Series(False, index=frame.index)
    limited = frame["phase2_original_text"].map(limit_reasoning_text)
    frame["phase2_original_text"] = limited.map(lambda pair: pair[0])
    frame["phase2_input_was_truncated"] = previously_truncated | limited.map(lambda pair: pair[1])

    routed = frame[frame["phase1_routed"]].copy()
    if routed.empty:
        raise ValueError("No routed rows were found in the final Phase 1 export.")
    if routed["phase2_original_text"].str.strip().eq("").any():
        raise ValueError("At least one routed row has empty Phase 2 original text.")
    if EXPECTED_TOTAL_ROWS is not None and len(frame) != EXPECTED_TOTAL_ROWS:
        raise ValueError(f"Expected {EXPECTED_TOTAL_ROWS} total rows, found {len(frame)}.")
    if EXPECTED_ROUTED_ROWS is not None and len(routed) != EXPECTED_ROUTED_ROWS:
        raise ValueError(f"Expected {EXPECTED_ROUTED_ROWS} routed rows, found {len(routed)}.")

    if MAX_ROWS is not None:
        routed = routed.head(min(MAX_ROWS, len(routed))).copy()
    return frame.reset_index(drop=True), routed.reset_index(drop=True)


phase1_df, routed_df = normalize_phase1_export(RESOLVED_PHASE1_INPUT)
routed_id_hash = hashlib.sha256(
    "\n".join(sorted(routed_df["example_id"])).encode("utf-8")
).hexdigest()
print("Phase 1 rows:", len(phase1_df))
print("Routed rows selected for this run:", len(routed_df))
print("Routed example_id SHA-256:", routed_id_hash)

if MAX_ROWS is None and EXPECTED_ROUTED_ID_HASH:
    if routed_id_hash != EXPECTED_ROUTED_ID_HASH:
        raise ValueError(
            f"Routed example_id hash {routed_id_hash} does not match the expected "
            f"{EXPECTED_ROUTED_ID_HASH}. This is a different routed set from the one the "
            "final Direct / CoT / SELF-DISCOVER comparisons were produced on, so nothing "
            "measured here would be comparable to them."
        )
    print("Routed set verified against the recorded 05_2 run.")

routed_phase1_errors = int((routed_df["phase1_label"] != routed_df["target_label"]).sum())
print(
    f"Routed Phase 1 errors (E_R): {routed_phase1_errors} "
    f"({routed_phase1_errors / len(routed_df):.2%} of routed); "
    f"conditional oracle end-to-end: "
    f"{(phase1_df['phase1_label'].eq(phase1_df['target_label']).mean() + routed_phase1_errors / len(phase1_df)):.4%}"
)
print("Routed rows with truncated Phase 2 input:",
      int(routed_df["phase2_input_was_truncated"].sum()))
display(routed_df[["example_id", "target_label", "phase1_label", "phase1_confidence"]].head())

## Model, generation, persistence, and evaluation helpers

In [ ]:
# Carried over from 05_2 unchanged, so scoring is identical across notebooks.

def stable_seed(example_id, method, stage):
    key = f"{BASE_SEED}:{example_id}:{method}:{stage}".encode("utf-8")
    return int.from_bytes(hashlib.sha256(key).digest()[:4], "big")


def set_generation_seed(seed):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_chat_model(model_name, model_revision):
    tokenizer = AutoTokenizer.from_pretrained(
        model_name, revision=model_revision, trust_remote_code=True,
        token=os.environ.get("HF_TOKEN")
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"
    quantization_config = None
    if LOAD_IN_4BIT:
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_use_double_quant=True,
        )
    model = AutoModelForCausalLM.from_pretrained(
        model_name,
        revision=model_revision,
        torch_dtype=torch.float16,
        device_map="auto",
        quantization_config=quantization_config,
        trust_remote_code=True,
        token=os.environ.get("HF_TOKEN"),
    )
    resolved_revision = getattr(model.config, "_commit_hash", None)
    if resolved_revision and resolved_revision != model_revision:
        raise ValueError(
            f"Resolved model revision {resolved_revision} differs from frozen {model_revision}."
        )
    model.eval()
    return tokenizer, model


def format_messages_without_chat_template(messages):
    system_parts = [m["content"] for m in messages if m.get("role") == "system"]
    dialogue = [m for m in messages if m.get("role") != "system"]
    system_text = "\n".join(system_parts).strip()
    prompt, pending_user, first_turn = "", None, True
    for message in dialogue:
        role = message.get("role")
        content = str(message.get("content", "")).strip()
        if role == "user":
            pending_user = content
        elif role == "assistant" and pending_user is not None:
            if first_turn and system_text:
                prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST] {content} </s>"
            else:
                prompt += f"<s>[INST] {pending_user} [/INST] {content} </s>"
            pending_user, first_turn = None, False
    if pending_user is not None:
        if first_turn and system_text:
            prompt += f"<s>[INST] <<SYS>>\n{system_text}\n<</SYS>>\n\n{pending_user} [/INST]"
        else:
            prompt += f"<s>[INST] {pending_user} [/INST]"
    return prompt


def build_chat_inputs(tokenizer, model, messages):
    if getattr(tokenizer, "chat_template", None):
        prompt = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    else:
        prompt = format_messages_without_chat_template(messages)
    encoded = tokenizer(prompt, return_tensors="pt")
    return {key: value.to(model.device) for key, value in encoded.items()}


def chat_generate(
    tokenizer, model, messages, *, max_new_tokens, seed,
    do_sample=False, temperature=None, top_p=None,
):
    set_generation_seed(seed)
    model_inputs = build_chat_inputs(tokenizer, model, messages)
    input_ids = model_inputs["input_ids"]
    terminators = [tokenizer.eos_token_id]
    if "<|eot_id|>" in tokenizer.get_vocab():
        eot_id = tokenizer.convert_tokens_to_ids("<|eot_id|>")
        if isinstance(eot_id, int) and eot_id >= 0 and eot_id not in terminators:
            terminators.append(eot_id)
    kwargs = dict(
        **model_inputs,
        max_new_tokens=max_new_tokens,
        eos_token_id=terminators,
        pad_token_id=tokenizer.eos_token_id,
        do_sample=do_sample,
    )
    if do_sample and temperature is not None:
        kwargs["temperature"] = temperature
    if do_sample and top_p is not None:
        kwargs["top_p"] = top_p
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        outputs = model.generate(**kwargs)
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    generated = outputs[0][input_ids.shape[-1]:]
    return {
        "text": tokenizer.decode(generated, skip_special_tokens=True).strip(),
        "input_tokens": int(input_ids.shape[-1]),
        "generated_tokens": int(generated.shape[-1]),
        "elapsed_seconds": float(elapsed),
        "hit_token_limit": bool(generated.shape[-1] >= max_new_tokens),
    }


def parse_final_label(output):
    text = str(output)
    match = re.search(
        r"Final\s+label\s*\*{0,2}\s*:\s*\*{0,2}\s*"
        r"(Depression|Neutral|Happy)\b",
        text,
        flags=re.I,
    )
    if match:
        return normalize_label(match.group(1))
    labels = {
        normalize_label(label)
        for label in re.findall(r"\b(Depression|Neutral|Happy)\b", text, flags=re.I)
    }
    return next(iter(labels)) if len(labels) == 1 else np.nan


def parse_provisional_label(output):
    '''Independent verdict recorded before the Phase 1 label is disclosed.'''
    text = str(output)
    match = re.search(
        r"Provisional\s+label\s*\*{0,2}\s*:\s*\*{0,2}\s*"
        r"(Depression|Neutral|Happy)\b",
        text,
        flags=re.I,
    )
    if match:
        return normalize_label(match.group(1))
    return parse_final_label(text)


def load_results(path):
    if path.exists() and path.stat().st_size > 0:
        frame = pd.read_csv(path)
        return frame.drop_duplicates(subset=["example_id"], keep="last")
    return pd.DataFrame()


def save_result(path, row):
    path.parent.mkdir(parents=True, exist_ok=True)
    existing = load_results(path)
    updated = pd.concat([existing, pd.DataFrame([row])], ignore_index=True)
    updated = updated.drop_duplicates(subset=["example_id"], keep="last")
    temporary = path.with_suffix(".tmp.csv")
    updated.to_csv(temporary, index=False)
    os.replace(temporary, path)


def clear_model(tokenizer, model):
    del tokenizer, model
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def summarize_method(full_phase1, routed_results, prediction_col, method):
    processed_ids = set(routed_results["example_id"].astype(str))
    evaluation_base = full_phase1
    scope = "full_end_to_end"
    if MAX_ROWS is not None:
        evaluation_base = full_phase1[full_phase1["example_id"].isin(processed_ids)].copy()
        scope = "smoke_routed_subset"
    merged = evaluation_base.merge(
        routed_results[["example_id", prediction_col]],
        on="example_id", how="left", validate="one_to_one",
    )
    merged["final_label"] = merged["phase1_label"]
    routed_mask = merged["phase1_routed"]
    parsed_routed_mask = routed_mask & merged[prediction_col].isin(LABELS)
    merged.loc[parsed_routed_mask, "final_label"] = merged.loc[parsed_routed_mask, prediction_col]
    merged["phase2_parse_success"] = ~routed_mask | parsed_routed_mask
    merged["phase2_fallback_to_phase1"] = routed_mask & ~parsed_routed_mask
    merged["phase1_correct"] = merged["phase1_label"].eq(merged["target_label"])
    merged["final_correct"] = merged["final_label"].eq(merged["target_label"])
    corrected = int((routed_mask & ~merged["phase1_correct"] & merged["final_correct"]).sum())
    introduced = int((routed_mask & merged["phase1_correct"] & ~merged["final_correct"]).sum())
    parsed = routed_results[prediction_col].isin(LABELS)
    summary = {
        "method": method,
        "scope": scope,
        "total_rows": len(merged),
        "routed_rows": int(routed_mask.sum()),
        "parsed_routed_rows": int(parsed.sum()),
        "parse_failures": int((~parsed).sum()),
        "phase1_accuracy": float(merged["phase1_correct"].mean()),
        "end_to_end_accuracy": float(merged["final_correct"].mean()),
        "end_to_end_macro_f1": float(
            f1_score(merged["target_label"], merged["final_label"], labels=LABELS, average="macro")
        ),
        "corrected": corrected,
        "introduced": introduced,
        "net_corrections": corrected - introduced,
    }
    return merged, summary

## Shared task policy

`CLASSIFICATION_POLICY` is copied verbatim from 05_2 so every variant classifies against the same
written label definitions as Direct, CoT and final SELF-DISCOVER.

In [ ]:
MODEL_NAME = ACTIVE_MODEL["name"]
MODEL_REVISION = ACTIVE_MODEL["revision"]
SD_ADVANCED_OUTPUT_DIR = DRIVE_OUTPUT_ROOT / ACTIVE_MODEL["folder"]
SD_ADVANCED_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# The discovered structure is task-level: it never saw an instance or a dataset, so both datasets
# share one artifact. Keeping it outside the per-dataset folder guarantees they use the same plan
# and makes "no per-dataset prompt tuning" checkable.
# Per model, shared across datasets: the structure is discovered by this model on a task
# description that names no dataset, so both datasets reuse it, but a different model must
# discover its own.
SHARED_STRUCTURE_DIR = (
    UNIFIED_FINAL_ROOT / "phase2" / MODEL_CONFIG[STRUCTURE_SOURCE_MODEL]["folder"]
)
if STRUCTURE_SOURCE_MODEL != MODEL:
    print(f"NOTE: running {MODEL} on the structure discovered by {STRUCTURE_SOURCE_MODEL}.")
SHARED_STRUCTURE_DIR.mkdir(parents=True, exist_ok=True)
LLAMA3_MAX_NEW_TOKENS = ACTIVE_MODEL["max_new_tokens"]
DISCOVERY_MAX_NEW_TOKENS = min(768, ACTIVE_MODEL["max_new_tokens"])

CLASSIFICATION_POLICY = """Classification Policy:
- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.
- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.
- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.

Assess the dominant emotional meaning of the full text. For mixed or shifting emotions, consider the overall trajectory and final takeaway. Do not decide from isolated words, and do not make clinical diagnoses or treatment recommendations."""

ANNOTATOR_SYSTEM_PROMPT = (
    "You are an expert annotator for research-oriented, non-clinical emotion classification. "
    "Do not provide clinical diagnosis, medical inference, treatment advice, or professional "
    "mental health advice."
)

In [ ]:
select_prompt = """
In order to solve the given task:
<Task>
{Task}
</Task>
Select several modules that are crucial for solving the tasks above
from all the reasoning module description given below:
{resonining_modules}
"""

adapt_prompt = """
Rephrase and specify each reasoning module so that it better helps solving the task:
<Task>
{Task}
</Task>
SELECTED module descriptions:
{selected_modules}
Adapt each reasoning module description to better solve the task:
"""


## Final task-level components

The task policy, 18 emotion-oriented reasoning modules, and evidence requirements are shared by the final v6c protocol.


In [ ]:
EMOTION_REASONING_MODULES = """
1 Separate the writer's own current emotional state from emotions they attribute to other people, to quoted text, or to hypothetical situations.
2 Locate the emotional stance the post opens with and the emotional stance it closes with, and state explicitly whether they differ.
3 Determine the final emotional takeaway the writer leaves the reader with, instead of averaging every emotional word in the post.
4 Distinguish a past state the writer describes as changed or resolved from a state that is still ongoing at the time of writing.
5 Decide whether the post is primarily informational, procedural, advice-giving, or a question about a topic, rather than an expression of the writer's own current feelings.
6 Check for irony, sarcasm, dark humour, or understatement that inverts the literal sentiment of a phrase.
7 Weigh explicit emotional statements against implicit signals such as sleep, appetite, isolation, disrupted routine, or loss of interest.
8 Identify markers of resolution: relief, gratitude, recovery, accomplishment, reconnection, or a plan the writer feels good about.
9 Identify markers of unresolved distress: hopelessness, exhaustion, self-devaluation, numbness, withdrawal, or an appeal for help that the post never resolves.
10 Judge emotional intensity, separating strong affect from mild, passing, or conventional affect.
11 Check whether the post seeks support or asks a question, and whether the surrounding text is distressed or matter-of-fact about it.
12 Separate discussing mental health as a subject, or reporting facts about it, from the writer expressing their own mental state.
13 For long or multi-topic posts, identify which segment carries the writer's main point and weight that segment above digressions.
14 Rank all three candidate labels for this post and name which label is second-best, rather than stopping at the first label that seems to fit.
15 State the single strongest piece of textual evidence for each of the three labels before choosing between them.
16 Test the leading label by asking what would have to appear in the text for a different label to be correct, then check whether it appears.
17 Check whether the tone is measured and observational rather than emotionally invested, which can occur even when the subject matter is heavy.
18 Check whether apparent positivity is aspirational, hypothetical, conventional politeness, or directed at someone else, rather than reflecting the writer's own current state."""

TASK_LEVEL_DESCRIPTION = """
Task: research-oriented, non-clinical emotion annotation of English social-media posts.

Each instance is one post, consisting of a title and a body, written by a single author. An annotator
must assign exactly one label from Depression, Neutral, Happy, according to the following policy.

{policy}

The posts are real, user-written text. They vary greatly in length. A single post may cover several
topics, quote or paraphrase other people, ask a question, give advice, report factual information,
describe a past episode that has since resolved, or mix positive and negative cues within the same
paragraph. The annotator must decide the dominant emotional meaning of the post as a whole, and must
be able to point to the text that supports the decision.

No clinical diagnosis, medical inference, or treatment advice is produced. The output is a research
annotation label only.
"""

def build_task_level_description():
    return TASK_LEVEL_DESCRIPTION.replace("{policy}", CLASSIFICATION_POLICY)

def stage_totals(stages):
    return {
        "input_tokens_total": sum(item["input_tokens"] for item in stages),
        "generated_tokens_total": sum(item["generated_tokens"] for item in stages),
        "generation_seconds_total": sum(item["elapsed_seconds"] for item in stages),
        "any_stage_hit_token_limit": any(item["hit_token_limit"] for item in stages),
    }


## Final per-post execution prompt

The cached compact plan is applied once to each routed post with greedy decoding and no Phase 1 label disclosure.


In [ ]:
V6_MAX_NEW_TOKENS = ACTIVE_MODEL["v6_max_new_tokens"]

V6_TEMPLATE = """Annotate one social-media post for a research-oriented, non-clinical emotion study.

{policy}

Reasoning structure to follow:
{structure}

Post:
<post>
{text}
</post>

Work through the reasoning structure on this post. For every step, quote the exact words from the
post that your finding rests on. Do not decide the label before the final step.

Time frame and whose state it is. Before weighing anything, establish two things and quote the
evidence for each:
- Is the emotional state the writer reports about their present, or about a past they are describing
  from a changed position?
- Is it the writer's own state, or a state they attribute to someone else, to a quotation, or to a
  hypothetical?
These cut in every direction. A post can describe severe hardship and still be Happy when the writer
presents it as past and resolved. A post can be light or matter-of-fact in tone and still be
Depression when the distress is present and unresolved. A post can recount someone else's crisis
while the writer is composed, which is Neutral.

Evidence ledger. Search just as hard for each of the three labels; do not give a shorter treatment to
the ones you find less likely.

Depression - strongest supporting quote: "..." | strongest quote against: "..."
Neutral - strongest supporting quote: "..." | strongest quote against: "..."
Happy - strongest supporting quote: "..." | strongest quote against: "..."

Counterfactual check. Name your second-best label. State what would have to appear in this post for
that label to be correct instead, then say whether the post contains it.

Then end with exactly one line, in this form:
Final label: Depression
or Final label: Neutral
or Final label: Happy"""


def build_v6_prompt(structure_text, post_text):
    return (
        V6_TEMPLATE
        .replace("{policy}", CLASSIFICATION_POLICY)
        .replace("{structure}", str(structure_text).strip())
        .replace("{text}", str(post_text))
    )


def run_v6_max_prompt(tokenizer, model, row, task_structure=None):
    result = chat_generate(
        tokenizer, model,
        [{"role": "system", "content": ANNOTATOR_SYSTEM_PROMPT},
         {"role": "user", "content": build_v6_prompt(
             task_structure["reasoning_structure"], row["phase2_original_text"]
         )}],
        max_new_tokens=V6_MAX_NEW_TOKENS,
        seed=stable_seed(row["example_id"], "v6_max_prompt", "single"),
        do_sample=False,
    )
    label = parse_final_label(result["text"])
    return {
        "sd_selected_modules": task_structure["selected_modules"],
        "sd_adapted_modules": task_structure["adapted_modules"],
        "sd_reasoning_structure": task_structure["reasoning_structure"],
        "sd_applied_reasoning": result["text"],
        "sd_final_answer": result["text"],
        # No Phase 1 label is ever shown, so the single verdict is both provisional and final.
        "provisional_label": label,
        "final_label": label,
        "vote_labels": "",
        "vote_unanimous": np.nan,
        **stage_totals([result]),
    }

## Compact plan construction and reuse

SELECT, ADAPT, and IMPLEMENT produce at most three task-level steps. The released model-specific plan is loaded below for an exact rerun and reused across both datasets.


In [ ]:
IMPLEMENT_COMPACT_PROMPT = """
Operationalize the reasoning modules into a step-by-step reasoning plan in JSON format.

<Task>
{Task}
</Task>

ADAPTED module descriptions:
{adapted_modules}

Implement a reasoning structure that generalises to every instance of this task.

The plan must obey these constraints. They exist because a plan that violates them throws away
evidence before it can be used, or is too long to be carried out:

1. **At most three steps.** Merge related checks rather than listing them separately.
2. **Each step is one line.** A name and a single sentence, nothing more. No nested fields, no
   examples, no restating of the instructions.
3. Every step takes the full original post as its input. A step may refer to an earlier step's
   finding, but it must re-read the post rather than work only from an earlier summary.
4. No step before the final one may name, choose, rank or imply the answer.
5. The final step is the only one that weighs the gathered evidence and selects a single label.

Return only the JSON plan.
"""

TASK_LEVEL_STRUCTURE_COMPACT_PATH = (
    SHARED_STRUCTURE_DIR / "task_level_reasoning_structure_compact.json"
)


def discover_compact_structure(tokenizer, model, *, force=False):
    """Stage 1 with the plan length capped, for a model that cannot carry a long one.

    Llama 2 walked only 40.7% of the four steps it wrote for itself, and the unexecuted steps still
    occupied the prompt. Capping the plan at three one-line steps is meant to make completion
    reachable while keeping the discovery stage intact, so this is still SELF-DISCOVER rather than a
    hand-written plan.
    """
    if STRUCTURE_SOURCE_MODEL != MODEL:
        if not TASK_LEVEL_STRUCTURE_COMPACT_PATH.exists():
            raise FileNotFoundError(
                f"STRUCTURE_SOURCE_MODEL={STRUCTURE_SOURCE_MODEL!r} but no compact structure is "
                f"cached at {TASK_LEVEL_STRUCTURE_COMPACT_PATH}."
            )
        borrowed = json.loads(TASK_LEVEL_STRUCTURE_COMPACT_PATH.read_text())
        print(f"Borrowing the compact structure discovered by {STRUCTURE_SOURCE_MODEL}.")
        return borrowed
    if TASK_LEVEL_STRUCTURE_COMPACT_PATH.exists() and not force:
        cached = json.loads(TASK_LEVEL_STRUCTURE_COMPACT_PATH.read_text())
        print("Reusing cached compact reasoning structure.")
        return cached

    task = build_task_level_description()
    select = select_prompt.replace("{Task}", task).replace(
        "{resonining_modules}", EMOTION_REASONING_MODULES
    )
    selected = chat_generate(
        tokenizer, model, [{"role": "user", "content": select}],
        max_new_tokens=DISCOVERY_MAX_NEW_TOKENS,
        seed=stable_seed("TASK_LEVEL_COMPACT", "compact", "select"), do_sample=False,
    )
    adapt = adapt_prompt.replace("{Task}", task).replace("{selected_modules}", selected["text"])
    adapted = chat_generate(
        tokenizer, model, [{"role": "user", "content": adapt}],
        max_new_tokens=DISCOVERY_MAX_NEW_TOKENS,
        seed=stable_seed("TASK_LEVEL_COMPACT", "compact", "adapt"), do_sample=False,
    )
    implement = IMPLEMENT_COMPACT_PROMPT.replace("{Task}", task).replace(
        "{adapted_modules}", adapted["text"]
    )
    structure = chat_generate(
        tokenizer, model, [{"role": "user", "content": implement}],
        max_new_tokens=DISCOVERY_MAX_NEW_TOKENS,
        seed=stable_seed("TASK_LEVEL_COMPACT", "compact", "implement"), do_sample=False,
    )
    payload = {
        "model_name": MODEL_NAME,
        "module_bank": "emotion_domain_v1_18_modules",
        "implement_prompt": "compact_v1_max_3_steps",
        "task_description": task,
        "selected_modules": selected["text"],
        "adapted_modules": adapted["text"],
        "reasoning_structure": structure["text"],
        "hit_token_limit": any(
            item["hit_token_limit"] for item in (selected, adapted, structure)
        ),
    }
    TASK_LEVEL_STRUCTURE_COMPACT_PATH.write_text(json.dumps(payload, indent=2))
    steps = len(re.findall(r'"(?:module|name)"\s*:', structure["text"]))
    print("Discovered and cached a new compact reasoning structure.")
    print(f"  steps: {steps} (asked for at most 3), "
          f"{len(structure['text'])} characters "
          f"(the non-compact plan was 2,911)")
    if steps > 3:
        print("  NOTE: the model exceeded the three-step cap; the constraint was not honoured.")
    return payload

In [ ]:
# Exact model-specific plans used for the reported v6c runs.
USE_RELEASE_CACHED_PLAN = True
RELEASE_CACHED_PLANS = {
  "llama2": {
    "model_name": "NousResearch/Llama-2-7b-chat-hf",
    "module_bank": "emotion_domain_v1_18_modules",
    "implement_prompt": "compact_v1_max_3_steps",
    "task_description": "\nTask: research-oriented, non-clinical emotion annotation of English social-media posts.\n\nEach instance is one post, consisting of a title and a body, written by a single author. An annotator\nmust assign exactly one label from Depression, Neutral, Happy, according to the following policy.\n\nClassification Policy:\n- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.\n- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.\n- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.\n\nAssess the dominant emotional meaning of the full text. For mixed or shifting emotions, consider the overall trajectory and final takeaway. Do not decide from isolated words, and do not make clinical diagnoses or treatment recommendations.\n\nThe posts are real, user-written text. They vary greatly in length. A single post may cover several\ntopics, quote or paraphrase other people, ask a question, give advice, report factual information,\ndescribe a past episode that has since resolved, or mix positive and negative cues within the same\nparagraph. The annotator must decide the dominant emotional meaning of the post as a whole, and must\nbe able to point to the text that supports the decision.\n\nNo clinical diagnosis, medical inference, or treatment advice is produced. The output is a research\nannotation label only.\n",
    "selected_modules": "Of the reasoning modules listed above, the following are the most crucial for solving the task of research-oriented, non-clinical emotion annotation of English social-media posts:\n\n1. Separate the writer's own current emotional state from emotions they attribute to other people, to quoted text, or to hypothetical situations.\n2. Locate the emotional stance the post opens with and the emotional stance it closes with, and state explicitly whether they differ.\n3. Determine the final emotional takeaway the writer leaves the reader with, instead of averaging every emotional word in the post.\n4. Distinguish a past state the writer describes as changed or resolved from a state that is still ongoing at the time of writing.\n5. Identify markers of resolution: relief, gratitude, recovery, accomplishment, reconnection, or a plan the writer feels good about.\n6. Identify markers of unresolved distress: hopelessness, exhaustion, self-devaluation, numbness, withdrawal, or an appeal for help that the post never resolves.\n7. Judge emotional intensity, separating strong affect from mild, passing, or conventional affect.\n8. Check whether the post seeks support or asks a question, and whether the surrounding text is distressed or matter-of-fact about it.\n9. Separate discussing mental health as a subject, or reporting facts about it, from the writer expressing their own mental state.\n10. For long or multi-topic posts, identify which segment carries the writer's main point and weight that segment above digressions.\n\nThese modules are crucial because they help the annotator accurately determine the dominant emotional meaning of the post, including the writer's own emotional state, the emotional stance of the post, and the final emotional takeaway. By separating the writer's emotions from those of others, identifying markers of resolution and unresolved distress, and judging emotional intensity, the annotator can make an informed decision about the most appropriate label for the post.",
    "adapted_modules": "Sure! Here are rephrased and specified versions of the reasoning module descriptions to better help solve the task of research-oriented, non-clinical emotion annotation of English social-media posts:\n\n1. Separate the writer's own current emotional state from emotions they attribute to other people, to quoted text, or to hypothetical situations:\n\nTo accurately annotate the emotional meaning of a post, it's essential to distinguish the writer's own emotions from those they describe in relation to others, quoted text, or hypothetical situations. This involves identifying the emotional language used in the post and determining whether it refers to the writer's own emotional state or someone else's. By separating the writer's emotions from those of others, the annotator can ensure that the label assigned to the post accurately reflects the dominant emotional meaning.\n\n2. Locate the emotional stance the post opens with and the emotional stance it closes with, and state explicitly whether they differ:\n\nThe emotional stance of a post can provide valuable insight into the writer's emotional state and the overall tone of the post. By locating the emotional stance the post opens with and the emotional stance it closes with, the annotator can determine whether these stances differ and how they contribute to the overall emotional meaning of the post. This information can help the annotator assign the most appropriate label to the post.\n\n3. Determine the final emotional takeaway the writer leaves the reader with, instead of averaging every emotional word in the post:\n\nRather than averaging every emotional word in the post, the annotator should focus on determining the final emotional takeaway the writer leaves the reader with. This involves identifying the emotional language used in the post's conclusion and determining whether it reflects a positive, negative, or neutral emotional state. By focusing on the final takeaway, the annotator can ensure that the label assigned to the post accurately reflects the dominant emotional meaning.\n\n4. Distinguish a past state the writer describes as changed or resolved from a state that is still ongoing at the time of writing:\n\nIn some cases, the post may describe a past state that has since resolved or changed. To accurately annotate the emotional",
    "reasoning_structure": "Here is a step-by-step reasoning plan in JSON format to annotate the emotional meaning of English social-media posts:\n\nStep 1: Separate the writer's own emotions from those of others (input: full post)\n\n* Identify emotional language used in the post (e.g. \"I feel\", \"they feel\", \"it seems\")\n* Determine whether the language refers to the writer's own emotions or someone else's\n* Assign a label based on the identified emotional language (e.g. \"Depression\" if the language refers to the writer's own emotions, \"Neutral\" if it refers to someone else's emotions)\n\nStep 2: Locate the emotional stance the post opens with and the emotional stance it closes with (input: full post)\n\n* Identify the opening and closing sentences of the post\n* Determine whether the emotional stance of the opening sentence differs from the emotional stance of the closing sentence\n* Assign a label based on the identified emotional stance (e.g. \"Depression\" if the stance differs, \"Neutral\" if the stance is the same)\n\nStep 3: Determine the final emotional takeaway the writer leaves the reader with (input: full post)\n\n* Identify the concluding sentence of the post\n* Determine whether the concluding sentence reflects a positive, negative, or neutral emotional state\n* Assign a label based on the identified emotional takeaway (e.g. \"Happy\" if the takeaway is positive, \"Neutral\" if it is neutral, \"Depression\" if it is negative)\n\nStep 4 (final step): Weigh the gathered evidence and select a label (input: full post)\n\n* Review the labels assigned in steps 1-3\n* Select the label that best reflects the dominant emotional meaning of the post\n* Output the selected label as the research annotation label.\n\nNote: These steps are designed to be carried out in sequence, with each step building on the previous one to gather more information and refine the annotation. The final step is the only one that assigns a label to the post, based on the gathered evidence.",
    "hit_token_limit": True
  },
  "llama3": {
    "model_name": "NousResearch/Meta-Llama-3-8B-Instruct",
    "module_bank": "emotion_domain_v1_18_modules",
    "implement_prompt": "compact_v1_max_3_steps",
    "task_description": "\nTask: research-oriented, non-clinical emotion annotation of English social-media posts.\n\nEach instance is one post, consisting of a title and a body, written by a single author. An annotator\nmust assign exactly one label from Depression, Neutral, Happy, according to the following policy.\n\nClassification Policy:\n- Depression: unresolved sadness, hopelessness, emotional distress, emotional exhaustion, withdrawal, self-devaluation, or a clearly negative overall trajectory is dominant.\n- Neutral: the text is mainly factual, routine, balanced, informational, or emotionally mild, without a dominant positive or distress-related state.\n- Happy: happiness, relief, gratitude, accomplishment, fulfillment, or a clearly positive resolution is dominant.\n\nAssess the dominant emotional meaning of the full text. For mixed or shifting emotions, consider the overall trajectory and final takeaway. Do not decide from isolated words, and do not make clinical diagnoses or treatment recommendations.\n\nThe posts are real, user-written text. They vary greatly in length. A single post may cover several\ntopics, quote or paraphrase other people, ask a question, give advice, report factual information,\ndescribe a past episode that has since resolved, or mix positive and negative cues within the same\nparagraph. The annotator must decide the dominant emotional meaning of the post as a whole, and must\nbe able to point to the text that supports the decision.\n\nNo clinical diagnosis, medical inference, or treatment advice is produced. The output is a research\nannotation label only.\n",
    "selected_modules": "Based on the task description, the following modules are crucial for solving the task:\n\n1. **Emotion Identification**: Separate the writer's own current emotional state from emotions they attribute to other people, to quoted text, or to hypothetical situations.\n2. **Emotional Trajectory**: Locate the emotional stance the post opens with and the emotional stance it closes with, and state explicitly whether they differ.\n3. **Emotional Takeaway**: Determine the final emotional takeaway the writer leaves the reader with, instead of averaging every emotional word in the post.\n4. **Contextual Analysis**: Distinguish a past state the writer describes as changed or resolved from a state that is still ongoing at the time of writing.\n5. **Purpose Identification**: Decide whether the post is primarily informational, procedural, advice-giving, or a question about a topic, rather than an expression of the writer's own current feelings.\n6. **Sentiment Inversion**: Check for irony, sarcasm, dark humor, or understatement that inverts the literal sentiment of a phrase.\n7. **Emotional Intensity**: Weigh explicit emotional statements against implicit signals such as sleep, appetite, isolation, disrupted routine, or loss of interest.\n8. **Resolution Identification**: Identify markers of resolution: relief, gratitude, recovery, accomplishment, reconnection, or a plan the writer feels good about.\n9. **Distress Identification**: Identify markers of unresolved distress: hopelessness, exhaustion, self-devaluation, numbness, withdrawal, or an appeal for help that the post never resolves.\n10. **Label Ranking**: Rank all three candidate labels for this post and name which label is second-best, rather than stopping at the first label that seems to fit.\n\nThese modules are essential for accurately annotating the emotional meaning of the social media posts, considering the writer's emotional state, the emotional trajectory of the post, and the context in which the emotions are expressed.",
    "adapted_modules": "Here are the adapted module descriptions:\n\n1. **Emotion Extraction**: Identify the writer's explicit and implicit emotional expressions, distinguishing between their own emotions and those attributed to others, quoted text, or hypothetical situations. Consider the context in which emotions are expressed to disentangle mixed or shifting emotions.\n\n2. **Emotional Arc**: Analyze the emotional trajectory of the post, identifying the emotional stance at the beginning and end of the text. Determine if the emotional tone shifts or remains consistent throughout, and note any significant changes or resolutions.\n\n3. **Emotional Essence**: Determine the dominant emotional takeaway the writer intends to convey, rather than simply averaging the emotional tone of the post. Consider the writer's purpose and the emotional resonance of the text as a whole.\n\n4. **Contextual Clarification**: Disentangle past events or experiences from the writer's current emotional state. Identify whether the writer is describing a resolved or ongoing situation, and consider the implications for their emotional expression.\n\n5. **Purpose Identification**: Determine the primary purpose of the post, such as informational, advisory, or questioning. This helps to contextualize the emotional expression and identify potential emotional cues.\n\n6. **Sentiment Reversal**: Detect instances of irony, sarcasm, dark humor, or understatement that invert the literal sentiment of a phrase. This module helps to uncover potential emotional contradictions or complexities.\n\n7. **Emotional Weight**: Evaluate the intensity and significance of explicit emotional statements, considering implicit signals such as sleep, appetite, isolation, or disrupted routine. This module helps to gauge the emotional impact of the post.\n\n8. **Resolution Detection**: Identify markers of resolution, such as relief, gratitude, recovery, accomplishment, reconnection, or a plan the writer feels good about. This module helps to determine if the post conveys a sense of closure or finality.\n\n9. **Distress Detection**: Identify markers of unresolved distress, including hopelessness, exhaustion, self-devaluation, numbness, withdrawal, or an appeal for help that remains unaddressed. This module helps to detect potential emotional pain or struggle.\n\n10. **Label Prioritization**: Rank the three candidate labels (Depression, Neutral, Happy) in order of likelihood, considering the emotional essence, emotional arc, and contextual factors. Provide a second-best label to ensure that the annotation is robust and defensible.\n\nThese adapted module descriptions aim to better capture the nuances of the task, focusing on the writer's emotional expression, the emotional trajectory of the post, and the context in which emotions are expressed.",
    "reasoning_structure": "Here is the operationalized reasoning plan in JSON format:\n\n```\n{\n  \"steps\": [\n    {\n      \"name\": \"Emotion Extraction\",\n      \"sentence\": \"Extract the writer's explicit and implicit emotional expressions, distinguishing between their own emotions and those attributed to others, quoted text, or hypothetical situations.\"\n    },\n    {\n      \"name\": \"Emotional Arc and Essence\",\n      \"sentence\": \"Analyze the emotional trajectory of the post, identifying the emotional stance at the beginning and end, and determine the dominant emotional takeaway the writer intends to convey.\"\n    },\n    {\n      \"name\": \"Contextual Clarification and Purpose Identification\",\n      \"sentence\": \"Disentangle past events or experiences from the writer's current emotional state, identify the primary purpose of the post, and consider the implications for their emotional expression.\"\n    }\n  ],\n  \"final_step\": {\n    \"name\": \"Label Prioritization\",\n    \"sentence\": \"Rank the three candidate labels (Depression, Neutral, Happy) in order of likelihood, considering the emotional essence, emotional arc, and contextual factors, and provide a second-best label to ensure the annotation is robust and defensible.\"\n  }\n}\n```\n\nThis plan consists of three main steps that extract the writer's emotional expressions, analyze the emotional trajectory and essence, and clarify the context and purpose of the post. The final step prioritizes the candidate labels based on the gathered evidence.",
    "hit_token_limit": False
  }
}

if USE_RELEASE_CACHED_PLAN:
    TASK_LEVEL_STRUCTURE_COMPACT_PATH.parent.mkdir(parents=True, exist_ok=True)
    TASK_LEVEL_STRUCTURE_COMPACT_PATH.write_text(
        json.dumps(RELEASE_CACHED_PLANS[MODEL], ensure_ascii=False, indent=2)
    )
    print(f"Loaded released cached plan for {MODEL}: {TASK_LEVEL_STRUCTURE_COMPACT_PATH}")


In [ ]:
VARIANT_CONFIG = {
    "v6c_compact": {
        "runner": run_v6_max_prompt,
        "prompt_version": "sd-v6c-compact-plan-max-3-steps",
        "needs_task_structure": "compact",
        "decoding": "greedy",
        "path": SD_ADVANCED_OUTPUT_DIR / "sd_v6c_compact_results.csv",
    }
}
RUN_VARIANTS = ["v6c_compact"]
per_example_generations = {"v6c_compact": 1}
print(f"Final release protocol: {RUN_VARIANTS[0]} | {MODEL} | {DATASET}")


## Run or resume the final v6c protocol

Results are saved row by row. Re-running this cell skips completed `example_id` values. Use `MAX_ROWS = 8` only for a separate smoke-test output directory; the final run uses `MAX_ROWS = None`.


In [ ]:
tokenizer, model = load_chat_model(MODEL_NAME, MODEL_REVISION)
structure = discover_compact_structure(tokenizer, model)
print("\n--- Released compact reasoning structure ---\n")
print(structure["reasoning_structure"])

config = VARIANT_CONFIG["v6c_compact"]
if not RESUME_FROM_EXISTING and config["path"].exists():
    config["path"].unlink()
existing = load_results(config["path"])
completed = set(existing.get("example_id", pd.Series(dtype=str)).astype(str))
pending = routed_df[~routed_df["example_id"].isin(completed)].copy()
print(f"v6c_compact: {len(completed)} completed, {len(pending)} pending")
for _, row in tqdm(pending.iterrows(), total=len(pending), desc="v6c_compact"):
    generated = run_v6_max_prompt(tokenizer, model, row, structure)
    result = {
        "example_id": row["example_id"],
        "target_label": row["target_label"],
        "phase1_label": row["phase1_label"],
        "phase1_confidence": row["phase1_confidence"],
        "phase2_original_text": row["phase2_original_text"],
        "phase2_input_was_truncated": row["phase2_input_was_truncated"],
        "method": "v6c_compact",
        "model_name": MODEL_NAME,
        "model_revision": MODEL_REVISION,
        "prompt_version": config["prompt_version"],
        "decoding": "greedy",
        "disclosed_phase1_confidence": False,
        "routed_id_hash": routed_id_hash,
        **generated,
    }
    save_result(config["path"], result)
clear_model(tokenizer, model)
print("Final SELF-DISCOVER v6c run finished or resumed.")


## Inspect unparsed rows

This inspection does not regenerate failures. Unparsed rows retain their Phase 1 label in the full-denominator export.


In [ ]:
inspect_variants = [name for name in VARIANT_CONFIG if VARIANT_CONFIG[name]["path"].exists()]
for variant in inspect_variants:
    results = load_results(VARIANT_CONFIG[variant]["path"])
    if results.empty:
        continue
    failed = results[~results["final_label"].isin(LABELS)]
    if failed.empty:
        print(f"{variant}: no unparsed rows")
        continue
    limit_flag = failed.get("any_stage_hit_token_limit")
    hit = int(limit_flag.fillna(False).astype(bool).sum()) if limit_flag is not None else 0
    print(f"\n=== {variant}: {len(failed)} unparsed rows, {hit} of them hit the token cap ===")
    for _, row in failed.iterrows():
        answer = str(row.get("sd_final_answer", ""))
        print(f"\n--- {row['example_id']} | generated_tokens="
              f"{row.get('generated_tokens_total')} | hit_cap={row.get('any_stage_hit_token_limit')}")
        print("    ...", answer[-400:].replace("\n", "\n    "))

## Throughput check and remaining-time projection

Run this at any point, including while the run cell is still going or after an interrupt. It reads
whatever rows are already on disk and projects the rest from **measured** throughput, so there is no
need to trust an a-priori estimate. Ten or fifteen completed rows of one variant are enough.

In [ ]:
throughput_rows = []
for variant in RUN_VARIANTS:
    config = VARIANT_CONFIG[variant]
    done = load_results(config["path"])
    if done.empty or "generation_seconds_total" not in done.columns:
        throughput_rows.append({"variant": variant, "completed": 0,
                                "remaining": len(routed_df), "status": "not started"})
        continue
    seconds = pd.to_numeric(done["generation_seconds_total"], errors="coerce").dropna()
    tokens = pd.to_numeric(done.get("generated_tokens_total"), errors="coerce").dropna()
    completed = len(seconds)
    remaining = max(0, len(routed_df) - completed)
    per_example = float(seconds.mean()) if completed else float("nan")
    throughput_rows.append({
        "variant": variant,
        "completed": completed,
        "remaining": remaining,
        "seconds_per_example": round(per_example, 1),
        "generated_tokens_per_example": round(float(tokens.mean()), 0) if len(tokens) else np.nan,
        "decode_tokens_per_second": (
            round(float(tokens.sum() / seconds.sum()), 1) if len(tokens) and seconds.sum() else np.nan
        ),
        "hours_done": round(float(seconds.sum()) / 3600, 2),
        "hours_remaining": round(per_example * remaining / 3600, 2) if completed else np.nan,
        "status": "complete" if remaining == 0 else "in progress",
    })

throughput_df = pd.DataFrame(throughput_rows)
display(throughput_df)

measured = throughput_df[throughput_df["completed"] > 0]
if not measured.empty:
    rate = measured["decode_tokens_per_second"].dropna()
    if len(rate):
        print(f"Measured decode throughput: {rate.mean():.1f} tokens/s")
    known = dict(zip(measured["variant"], measured["seconds_per_example"]))
    # Project unstarted variants from a started one, scaled by generations per example.
    reference = next(iter(known))
    per_generation = known[reference] / per_example_generations[reference]
    total_hours = 0.0
    for variant in RUN_VARIANTS:
        row = throughput_df[throughput_df["variant"] == variant].iloc[0]
        if row["completed"] > 0:
            total_hours += float(row["hours_remaining"] or 0)
        else:
            total_hours += (
                per_generation * per_example_generations[variant] * len(routed_df) / 3600
            )
    print(f"Projected time to finish every variant in RUN_VARIANTS: {total_hours:.1f} hours")
    print(
        "Colab disconnects before that on most plans. Re-run the setup cells and the run cell to "
        "resume; completed rows are skipped."
    )
else:
    print("No completed rows yet. Start the run cell, wait for ~10 rows, then re-run this cell.")

## Final full-denominator export


In [ ]:
results = load_results(VARIANT_CONFIG["v6c_compact"]["path"])
prediction_col = "self_discover_final_label"
scored = results.rename(columns={"final_label": prediction_col})
end_to_end, summary = summarize_method(
    phase1_df, scored, prediction_col, "SELF-DISCOVER"
)
end_to_end.to_csv(
    SD_ADVANCED_OUTPUT_DIR / "self_discover_v6c_end_to_end_predictions.csv",
    index=False,
)
pd.DataFrame([summary]).to_csv(
    SD_ADVANCED_OUTPUT_DIR / "self_discover_v6c_summary.csv",
    index=False,
)
display(pd.DataFrame([summary]))
print("Final v6c export complete. Parsing failures retained their Phase 1 labels.")
